# S1 pointer A: shuffle + curriculum 4-64 + temp floor (1x T4)

Gate vs the 1k A run: flips < 0.01, no accuracy drop (sel 0.858 / zs 0.900 on n=120).
1000 steps, 4000 rows, batch 4, Qwen3-0.6B-Base. Two forwards per step (order KL). About 90 min.

In [ ]:
import os, subprocess, sys, time, pathlib, json
REPO = "/kaggle/working/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "upp-p0", "https://github.com/nisaral/Soro-Boro.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "upp-p0"], check=False)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout, flush=True)
!pip -q install peft bitsandbytes huggingface_hub
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
t0 = time.time()
cmd = [
    sys.executable, "veyra/training/p0_upp_ablation.py",
    "--s1", "--configs", "A",
    "--base-model", "Qwen/Qwen3-0.6B-Base",
    "--train-rows", "4000", "--eval-rows", "120", "--batch", "4",
    "--max-seq-len", "832", "--max-steps", "1000", "--log-every", "50",
    "--out", "runs/s1_a",
]
print(" ".join(cmd), flush=True)
rc = subprocess.run(cmd).returncode
print(f"done in {(time.time()-t0)/60:.1f} min rc={rc}", flush=True)
if rc != 0:
    raise SystemExit(rc)
p = pathlib.Path("runs/s1_a/results.json")
print(p.read_text() if p.exists() else "no results")